# BasketCut AI V0.2 · Kaggle GPU 投篮候选研究

本 Notebook 用 **Torchvision Faster R-CNN COCO 运动球检测 + 篮筐 ROI + 时间聚类** 筛选篮筐附近的镜头。它**不识别进球是否命中，也不自动认球员**。成果作为待审 JSON 导入 Django。

## 运行前
1. Django 中打开该场比赛，暂停录像并框选篮筐，保存后下载 **GPU 配置 JSON**。
2. 将已获授权的比赛视频和对应配置 JSON 添加到 Kaggle **私有 Dataset**（不要公开发布）。
3. Notebook 设置 **Accelerator → GPU**。首次下载预训练权重需要 Notebook Internet 权限，注意账号额度和许可。
4. 按下面两个文件路径填入 Dataset 文件的真实路径；运行完成后下载 `/kaggle/working/candidates.json`。
5. 回 Django 对应比赛点“导入 Kaggle / GPU 篮筐附近候选”，人工逐条审核。

**版权/合规**：仅用于研发。请核查 PyTorch/Torchvision 及所用预训练权重的适用授权；使用真实球员影像须取得拍摄、分析和传播所需授权。免费 Kaggle GPU 不是持续运行的收费平台后端。


In [ ]:
import os, sys, subprocess
from pathlib import Path
print('Kaggle GPU setup')
try:
    import torch, torchvision, cv2
    print('torch:', torch.__version__, 'torchvision:', torchvision.__version__)
    print('CUDA:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none')
except ImportError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'opencv-python-headless', 'torchvision'], check=True)
    import torch, torchvision, cv2
    print('CUDA:', torch.cuda.is_available())

REPO_PATH = Path('/kaggle/working/shipingaoguang')
if not (REPO_PATH / 'highlights' / 'vision' / 'cli.py').exists():
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/mjw3177109/shipingaoguang.git', str(REPO_PATH)], check=True)
print('BasketCut repo:', REPO_PATH)


In [ ]:
# 修改为 Kaggle 左侧 Input 里真实的两个文件路径：
VIDEO_PATH = '/kaggle/input/your-private-basketball-data/game.mp4'
CONFIG_PATH = '/kaggle/input/your-private-basketball-data/game-1-hoops.json'
OUTPUT_PATH = '/kaggle/working/candidates.json'
INTERVAL_SECONDS = 1.5  # 越小越密集、越慢
MAX_FRAMES = 1800        # 首次研究限制计算量；超出会标记 incomplete_scan
DEVICE = 'auto'
assert Path(VIDEO_PATH).is_file(), '请先填写有效 VIDEO_PATH'
assert Path(CONFIG_PATH).is_file(), '请先填写有效 CONFIG_PATH'
assert (REPO_PATH / 'highlights/vision/cli.py').is_file()
args = [sys.executable, '-m', 'highlights.vision.cli', '--video', VIDEO_PATH,
        '--config', CONFIG_PATH, '--out', OUTPUT_PATH,
        '--interval', str(INTERVAL_SECONDS), '--max-frames', str(MAX_FRAMES), '--device', DEVICE]
subprocess.run(args, cwd=str(REPO_PATH), check=True)


In [ ]:
import json
from pathlib import Path
result = json.loads(Path(OUTPUT_PATH).read_text(encoding='utf-8'))
print(json.dumps(result['analysis'], ensure_ascii=False, indent=2))
print('候选数:', len(result['candidates']))
for item in result['candidates'][:15]:
    print(item['start_seconds'], '→', item['end_seconds'], item['note'])
if result['analysis']['incomplete_scan']:
    print('注意：本次只扫描视频前半部分！调整 MAX_FRAMES 或拆分试验。')
print('下载结果文件:', OUTPUT_PATH)
